# A6 — SVM Classification + Grid Optimization (WeakestLink endpoint)

Sprint issue (Lab 6):
- Consider SVM classification as an alternative to the classification endpoint
- Implement grid optimization for selecting the champion approach for this endpoint
- Challenge the current champion classification with the SVM approach :contentReference[oaicite:1]{index=1}

Scope of this notebook:
- WeakestLink **classification only** (regression handled by teammate)
- Train + tune an SVM (linear / RBF / poly) using **grid optimization** + CV
- Compare SVM vs **current champion classifier** (if available)
- Export the new champion artifacts if SVM wins

### Imports

In [1]:
import os
import json
import time
import warnings
from pathlib import Path
from datetime import datetime
from typing import Dict, Optional

import numpy as np
import pandas as pd

import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from mlflow.models.signature import infer_signature

from dotenv import load_dotenv
from os import getenv

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    classification_report,
)

warnings.filterwarnings("ignore")

### Paths (robust) + Load Data

In [2]:
SCORES_PATH   = Path("../../data/processed/20190108_scores_and_weak_links.csv")
FEATURES_PATH = Path("../../data/processed/AimoScore_WeakLink_big_scores.csv")

scores = pd.read_csv(SCORES_PATH)
big = pd.read_csv(FEATURES_PATH)

print("scores:", scores.shape)
print("big:", big.shape)
display(scores.head(2))
display(big.head(2))

scores: (2096, 17)
big: (2094, 43)


,ID,Date,SCORE,ForwardHead,LeftArmFallForward,RightArmFallForward,LeftShoulderElevation,RightShoulderElevation,ExcessiveForwardLean,LeftAsymmetricalWeightShift,RightAsymmetricalWeightShift,LeftKneeMovesInward,RightKneeMovesInward,LeftKneeMovesOutward,RightKneeMovesOutward,LeftHeelRises,RightHeelRises
0,cf0e95ab-d6ec-475a-9cc0-c9444b8dd080.Kinect,2018-07-19,0.612532,0.31,1.34,1.15,0.33,0.49,0.00,0.0,0.08,0.0,0.0,0.0,0.77,0.0,0.0
1,6210cc6d-575d-4e9f-bab1-4fe1da2a30a3.Kinect,2018-07-30,0.388809,1.49,1.42,1.39,0.00,0.00,1.08,0.1,0.00,0.0,0.0,0.0,0.40,0.0,0.0


,AimoScore,No_1_Angle_Deviation,No_2_Angle_Deviation,No_3_Angle_Deviation,No_4_Angle_Deviation,No_5_Angle_Deviation,No_6_Angle_Deviation,No_7_Angle_Deviation,No_8_Angle_Deviation,No_9_Angle_Deviation,...,No_20_NASM_Deviation,No_21_NASM_Deviation,No_22_NASM_Deviation,No_23_NASM_Deviation,No_24_NASM_Deviation,No_25_NASM_Deviation,No_1_Time_Deviation,No_2_Time_Deviation,EstimatedScore,ID
0,0.323667,0.538020,0.815878,0.346724,0.382114,0.302248,0.947872,0.275945,0.52176,0.457198,...,0.656624,0.642276,0.552846,0.648972,0.578192,0.560019,0.821616,0.818747,0.209947,0003cdcc-86ed-494a-a3b5-90d09e96e06b.Kinect
1,0.323699,0.443807,0.306552,0.823529,0.188905,0.497370,0.140124,0.664275,0.52176,0.729316,...,0.721186,0.826399,0.805356,0.848876,0.889048,0.816834,0.307987,0.248207,0.457198,003115c4-bdb8-491c-b571-8fcebdecf8ed.Kinect


### MLflow connection

In [3]:
load_dotenv("../../../../../env")

tracking_uri = getenv("MLFLOW_TRACKING_URI")
print("MLFLOW_TRACKING_URI =", tracking_uri)

if not tracking_uri:
    raise RuntimeError("MLFLOW_TRACKING_URI not set. Fix env path/variable.")

mlflow.set_tracking_uri(tracking_uri)

EXPERIMENT_NAME = "A6_svm_classification"
mlflow.set_experiment(EXPERIMENT_NAME)

client = MlflowClient()
print("Tracking:", mlflow.get_tracking_uri())
print("Experiment:", EXPERIMENT_NAME)

MLFLOW_TRACKING_URI = https://172.27.62.170:5001/
Tracking: https://172.27.62.170:5001/
Experiment: A6_svm_classification


### Helper functions (metrics + champion alignment)

In [4]:
def classification_metrics(y_true, y_pred, proba=None, classes=None) -> Dict[str, float]:
    out = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }
    if proba is not None:
        # only compute log_loss if shapes align
        try:
            labels_for_ll = classes if classes is not None else np.array(sorted(np.unique(y_true)), dtype=object)
            if proba.shape[1] == len(labels_for_ll):
                out["log_loss"] = float(log_loss(y_true, proba, labels=labels_for_ll))
            else:
                out["log_loss"] = float("nan")
        except Exception:
            out["log_loss"] = float("nan")
    return out

## Build WeakestLink labels (drop up to 5 classes + drop ties)

WeakestLink = weak link column with max score (Lab 3). :contentReference[oaicite:2]{index=2}  
We mimic the previous notebooks:
- Drop up to 5 weak link classes
- Detect ties (multiple maxima)
- Drop tie rows

In [5]:
# =========================
# A5b preprocessing EXACT (FIXED index alignment)
# =========================

DROP_WEAK_LINKS = {
    "LeftHeelRises",
    "RightHeelRises",
    "LeftKneeMovesInward",
    "RightAsymmetricalWeightShift",
}
DROP_TIES = True

NON_INDICATORS = {"ID", "Date", "SCORE"}
indicator_cols = [c for c in scores.columns if c not in NON_INDICATORS]
indicator_cols = [c for c in indicator_cols if c not in DROP_WEAK_LINKS]

scores[indicator_cols] = scores[indicator_cols].apply(pd.to_numeric, errors="coerce")

row_max = scores[indicator_cols].max(axis=1)
scores["HasTie"] = (scores[indicator_cols].eq(row_max, axis=0).sum(axis=1) > 1)
scores["WeakestLink"] = scores[indicator_cols].idxmax(axis=1)

labels = scores[["ID", "WeakestLink", "HasTie"]].dropna()

data = big.merge(labels, on="ID", how="inner")
print("Merged:", data.shape, "| tie rows:", int(data["HasTie"].sum()))

if DROP_TIES:
    data = data.loc[~data["HasTie"]].copy()
    print("After drop ties:", data.shape)

# ✅ CRITICAL: reset index ONCE so X/y share the same positional index
data = data.reset_index(drop=True)

y = data["WeakestLink"].astype(str)

# A5b feature selection: Angle + NASM only
angle_cols = [c for c in data.columns if c.startswith("No_") and c.endswith("_Angle_Deviation")]
nasm_cols  = [c for c in data.columns if c.startswith("No_") and c.endswith("_NASM_Deviation")]
feature_cols = sorted(angle_cols) + sorted(nasm_cols)

X = data[feature_cols].apply(pd.to_numeric, errors="coerce")

# A5b NaN policy: drop rows with any NaN (use positional mask)
mask = X.notna().all(axis=1).to_numpy()
X = X.loc[mask].reset_index(drop=True)
y = y.loc[mask].reset_index(drop=True)

print("Final X:", X.shape, "| classes:", y.nunique(), "| min class:", int(y.value_counts().min()))
display(y.value_counts().head(20))

Merged: (2094, 45) | tie rows: 370
After drop ties: (1724, 45)
Final X: (1724, 38) | classes: 10 | min class: 46


WeakestLink
RightArmFallForward            454
LeftArmFallForward             328
RightKneeMovesOutward          278
RightShoulderElevation         252
ExcessiveForwardLean            99
ForwardHead                     96
LeftAsymmetricalWeightShift     71
LeftShoulderElevation           51
LeftKneeMovesOutward            49
RightKneeMovesInward            46
Name: count, dtype: int64

## Join labels with predictors (via ID)

## Prepare X/y and drop singleton classes
We drop classes with <2 samples so split works.

## Train/test split + CV split guard
We auto-set n_splits so StratifiedKFold never breaks.

In [6]:
RANDOM_SEED = 42

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y
)

cv_splits = 5
cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_SEED)

print("X_train:", X_train.shape, "X_test:", X_test.shape, "cv_splits:", cv_splits)

X_train: (1379, 38) X_test: (345, 38) cv_splits: 5


## Load current champion from MLflow (instead of local file)

In [7]:
REGISTERED_MODEL_NAME = "weakest_link_model"

champion_run_id = None
champion_meta = None
champion_logged_metrics = None

def _try_download_meta(run_id: str) -> Optional[dict]:
    try:
        local_dir = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path="")
        p = Path(local_dir) / "champion_metadata.json"
        if p.exists():
            return json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        pass
    return None

# Resolve champion run_id via alias
try:
    mv = client.get_model_version_by_alias(REGISTERED_MODEL_NAME, "champion")
    champion_run_id = mv.run_id
    champion_meta = _try_download_meta(champion_run_id)
    champion_logged_metrics = client.get_run(champion_run_id).data.metrics
    print("Champion run_id:", champion_run_id)
except Exception as e:
    print("Could not resolve champion alias. Error:", repr(e))

print("Champion meta keys:", list(champion_meta.keys())[:12] if isinstance(champion_meta, dict) else None)
print("Champion logged metrics keys sample:", list(champion_logged_metrics.keys())[:15] if champion_logged_metrics else None)

# Feature alignment to champion if meta provides feature_cols
if champion_meta and "feature_cols" in champion_meta:
    feature_cols = champion_meta["feature_cols"]
    X_train = X_train.reindex(columns=feature_cols)
    X_test  = X_test.reindex(columns=feature_cols)
    print("Aligned to champion feature_cols:", len(feature_cols))
else:
    print("No champion meta feature_cols found; using A5b feature_cols.")

Champion run_id: 8f2f9018adb94b558ef66824887b0e3d
Champion meta keys: ['task', 'champion_model_key', 'sklearn_estimator', 'pipeline', 'created_utc', 'n_train', 'n_test', 'n_features', 'feature_cols', 'classes_dataset', 'classes_model', 'best_params']
Champion logged metrics keys sample: ['accuracy', 'balanced_accuracy', 'macro_f1', 'weighted_f1', 'log_loss']
Aligned to champion feature_cols: 38


### Set feature_cols + ALL_CLASSES from champion meta

### Align X_train/X_test to champion feature order

## SVM + Grid Optimization (CV)
Grid optimization with exponentially growing values for C and gamma, and normalize predictors. :contentReference[oaicite:3]{index=3}

60 minute run time

In [8]:
C_range = [2.0 ** p for p in range(-5, 11, 2)]          # 2^-5 .. 2^10
gamma_range = [2.0 ** p for p in range(-15, 4, 2)]      # 2^-15 .. 2^3

param_grid = [
    {
        "svc__kernel": ["linear"],
        "svc__C": C_range,
        "svc__class_weight": [None, "balanced"],
    },
    {
        "svc__kernel": ["rbf"],
        "svc__C": C_range,
        "svc__gamma": gamma_range + ["scale", "auto"],
        "svc__class_weight": [None, "balanced"],
    },
    {
        "svc__kernel": ["poly"],
        "svc__C": C_range,
        "svc__gamma": gamma_range + ["scale", "auto"],
        "svc__degree": [2, 3, 4, 5],
        "svc__coef0": [0.0, 0.5, 1.0, 2.0],
        "svc__class_weight": [None, "balanced"],
    },
    {
        "svc__kernel": ["sigmoid"],
        "svc__C": C_range,
        "svc__gamma": gamma_range + ["scale", "auto"],
        "svc__coef0": [0.0, 0.5, 1.0, 2.0],
        "svc__class_weight": [None, "balanced"],
    },
]

print("param_grid blocks:", len(param_grid))

param_grid blocks: 4


In [9]:
svm_pipe_fast = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("svc", SVC(probability=False, random_state=RANDOM_SEED)),
])

grid_fast = GridSearchCV(
    estimator=svm_pipe_fast,
    param_grid=param_grid,
    scoring="balanced_accuracy",
    n_jobs=-1,
    cv=cv,
    verbose=2,
    refit=True,
)

with mlflow.start_run(run_name="A6_SVM_gridsearch_FAST") as run:
    mlflow.set_tag("task", "classification")
    mlflow.set_tag("project_component", "weakest_link")
    mlflow.set_tag("model_key", "svm")
    mlflow.set_tag("primary_scoring", "balanced_accuracy")
    mlflow.set_tag("registered_model_name", REGISTERED_MODEL_NAME)

    mlflow.log_param("random_seed", RANDOM_SEED)
    mlflow.log_param("cv_folds", cv.get_n_splits())
    mlflow.log_param("dropped_weak_links", json.dumps(sorted(list(DROP_WEAK_LINKS))))

    t0 = time.perf_counter()
    grid_fast.fit(X_train, y_train)
    t1 = time.perf_counter()

    best_params = grid_fast.best_params_
    best_cv_balacc = float(grid_fast.best_score_)

    mlflow.log_metric("cv_best_balanced_accuracy", best_cv_balacc)
    mlflow.log_param("best_params", json.dumps(best_params, default=str))
    mlflow.log_metric("grid_fit_seconds", float(t1 - t0))

    print("Best CV balanced_accuracy:", best_cv_balacc)
    print("Best params:", best_params)
    print(f"GridSearch time: {t1 - t0:.2f}s")

Fitting 5 folds for each of 4048 candidates, totalling 20240 fits
Best CV balanced_accuracy: 0.5502441257297541
Best params: {'svc__C': 0.5, 'svc__class_weight': 'balanced', 'svc__coef0': 1.0, 'svc__degree': 3, 'svc__gamma': 0.0078125, 'svc__kernel': 'poly'}
GridSearch time: 313.90s
🏃 View run A6_SVM_gridsearch_FAST at: https://172.27.62.170:5001/#/experiments/11/runs/e3249dfb938040999118fc41be0be6df
🧪 View experiment at: https://172.27.62.170:5001/#/experiments/11


In [10]:
best_svm = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("svc", SVC(probability=True, random_state=RANDOM_SEED)),
])
best_svm.set_params(**best_params)

t0 = time.perf_counter()
best_svm.fit(X_train, y_train)
t1 = time.perf_counter()

print(f"Final SVM refit (probability=True) time: {t1 - t0:.2f}s")

Final SVM refit (probability=True) time: 0.32s


## Evaluate best SVM on test set

In [11]:
proba_svm = best_svm.predict_proba(X_test)
pred_svm = best_svm.predict(X_test)

ALL_CLASSES = sorted(pd.unique(y_test))  # safe for this split

svm_metrics = classification_metrics(
    y_test, pred_svm, proba=proba_svm, classes=np.array(ALL_CLASSES, dtype=object)
)

print("SVM test metrics:", svm_metrics)
print("\nClassification report:")
print(classification_report(y_test, pred_svm))

SVM test metrics: {'accuracy': 0.5420289855072464, 'balanced_accuracy': 0.5813124711545765, 'macro_f1': 0.4928274642089677, 'weighted_f1': 0.5529792961701832, 'log_loss': 1.0228992761850957}

Classification report:
                             precision    recall  f1-score   support

       ExcessiveForwardLean       0.40      0.70      0.51        20
                ForwardHead       0.35      0.74      0.47        19
         LeftArmFallForward       0.55      0.41      0.47        66
LeftAsymmetricalWeightShift       0.48      0.71      0.57        14
       LeftKneeMovesOutward       0.33      0.80      0.47        10
      LeftShoulderElevation       0.25      0.30      0.27        10
        RightArmFallForward       0.69      0.40      0.50        91
       RightKneeMovesInward       0.15      0.44      0.22         9
      RightKneeMovesOutward       0.98      0.89      0.93        56
     RightShoulderElevation       0.62      0.42      0.50        50

                   accur

## Compare SVM vs current champion

In [12]:
if champion_logged_metrics:
    # Prefer the champion's logged metrics (source of truth)
    champ_wf1 = champion_logged_metrics.get("weighted_f1") or champion_logged_metrics.get("test_weighted_f1")
    champ_acc = champion_logged_metrics.get("accuracy") or champion_logged_metrics.get("test_accuracy")
    champ_bal = champion_logged_metrics.get("balanced_accuracy") or champion_logged_metrics.get("test_balanced_accuracy")
    champ_mac = champion_logged_metrics.get("macro_f1") or champion_logged_metrics.get("test_macro_f1")
    champ_ll  = champion_logged_metrics.get("log_loss") or champion_logged_metrics.get("test_log_loss")

    champion_metrics = {
        "accuracy": float(champ_acc) if champ_acc is not None else None,
        "balanced_accuracy": float(champ_bal) if champ_bal is not None else None,
        "macro_f1": float(champ_mac) if champ_mac is not None else None,
        "weighted_f1": float(champ_wf1) if champ_wf1 is not None else None,
        "log_loss": float(champ_ll) if champ_ll is not None else None,
    }
    print("Champion metrics (logged):", champion_metrics)
else:
    champion_metrics = None
    print("Champion logged metrics unavailable; skipping logged comparison.")

Champion metrics (logged): {'accuracy': 0.663768115942029, 'balanced_accuracy': 0.5481671398776661, 'macro_f1': 0.5522275287286076, 'weighted_f1': 0.654381440617033, 'log_loss': 1.3875564256705957}


In [14]:
artifact_dir = "mlflow_svm_artifacts"
os.makedirs(artifact_dir, exist_ok=True)

# Save X_test and predictions
X_test_path = os.path.join(artifact_dir, "X_test_inputs.csv")
pd.DataFrame(X_test, columns=X_test.columns).to_csv(X_test_path, index=False)

pred_df = pd.DataFrame({
    "y_true": pd.Series(y_test).reset_index(drop=True).astype(str),
    "y_pred": pd.Series(pred_svm).reset_index(drop=True).astype(str),
})
proba_cols = [f"proba_{c}" for c in ALL_CLASSES]
pred_df = pd.concat([pred_df, pd.DataFrame(proba_svm, columns=proba_cols)], axis=1)

pred_path = os.path.join(artifact_dir, "test_outputs_predictions.csv")
pred_df.to_csv(pred_path, index=False)

meta = {
    "task": "WeakestLink classification (SVM)",
    "created_utc": datetime.utcnow().isoformat(),
    "random_seed": int(RANDOM_SEED),
    "cv_folds": int(cv.get_n_splits()),
    "drop_weak_links": sorted(list(DROP_WEAK_LINKS)),
    "drop_ties": bool(DROP_TIES),
    "feature_cols": list(X_train.columns),
    "classes_dataset": [str(c) for c in ALL_CLASSES],
    "best_params": {k: (float(v) if isinstance(v, (np.floating,)) else v) for k, v in best_params.items()},
    "metrics": svm_metrics,
}
meta_path = os.path.join(artifact_dir, "svm_metadata.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

with mlflow.start_run(run_name="A6_SVM_FINAL_log_and_register") as run:
    run_id = run.info.run_id

    mlflow.set_tag("task", "classification")
    mlflow.set_tag("project_component", "weakest_link")
    mlflow.set_tag("model_key", "svm")
    mlflow.set_tag("registered_model_name", REGISTERED_MODEL_NAME)

    mlflow.log_param("random_seed", RANDOM_SEED)
    mlflow.log_param("cv_folds", cv.get_n_splits())
    mlflow.log_param("best_params", json.dumps(best_params, default=str))

    mlflow.log_metrics({k: float(v) for k, v in svm_metrics.items() if v is not None and not (isinstance(v, float) and np.isnan(v))})

    mlflow.log_artifact(X_test_path)
    mlflow.log_artifact(pred_path)
    mlflow.log_artifact(meta_path)

    X_example = X_train.head(20)
    y_example = best_svm.predict(X_example)
    signature = infer_signature(X_example, y_example)

    # Log model to run artifacts
    model_info = mlflow.sklearn.log_model(
        sk_model=best_svm,
        artifact_path="model",
        signature=signature,
        input_example=X_example,
    )

print(" Logged SVM model to MLflow. Run ID:", run_id)

# ---- Optional: register + promote if SVM beats champion (by weighted_f1) ----
svm_wf1 = svm_metrics.get("weighted_f1", -np.inf)
champ_wf1 = champion_metrics.get("weighted_f1") if champion_metrics else None

if champ_wf1 is not None and svm_wf1 > champ_wf1:
    print(f"SVM beats champion by weighted_f1 ({svm_wf1:.4f} > {champ_wf1:.4f}). Registering & promoting...")

    # Register under weakest_link_model
    model_uri = f"runs:/{run_id}/model"
    mv = mlflow.register_model(model_uri, REGISTERED_MODEL_NAME)

    # Promote alias champion (or Production fallback)
    client = MlflowClient()
    version = mv.version

    promoted = False
    try:
        client.set_registered_model_alias(REGISTERED_MODEL_NAME, "champion", str(version))
        promoted = True
        print(f"Set alias 'champion' -> {REGISTERED_MODEL_NAME} v{version}")
    except Exception as e_alias:
        try:
            client.transition_model_version_stage(
                name=REGISTERED_MODEL_NAME,
                version=version,
                stage="Production",
                archive_existing_versions=True
            )
            promoted = True
            print(f"Transitioned {REGISTERED_MODEL_NAME} v{version} to Production.")
        except Exception as e_stage:
            print("Could not promote automatically.")
            print("Alias error:", repr(e_alias))
            print("Stage error:", repr(e_stage))
else:
    print("SVM did not beat champion by logged weighted_f1 (or champion metrics unavailable). Not promoting.")

2026/03/10 19:55:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run A6_SVM_FINAL_log_and_register at: https://172.27.62.170:5001/#/experiments/11/runs/e34dc338cbc243189e23b39f30406ed1
🧪 View experiment at: https://172.27.62.170:5001/#/experiments/11
 Logged SVM model to MLflow. Run ID: e34dc338cbc243189e23b39f30406ed1
SVM did not beat champion by logged weighted_f1 (or champion metrics unavailable). Not promoting.
